In [ ]:
import numpy as np
import pandas as pd
from pyproj import Transformer

from sklearn.model_selection import train_test_split , KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb


/usr/local/lib/python3.12/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:19: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -------------------- Config --------------------
CURRENT_YEAR_SHAMSI = 1404
TEST_SIZE = 0.2
RANDOM_SEED = 42
SAMPLE_LIMIT = 20_000

MODEL_TYPE = "xgb"          # "rf" or "xgb"
USE_QUANTILE = True
QUANTILE_TAU = 0.5          # for XGB (0.5 ~ median/MAE)

PRE_CLIP_Q = (0.025, 0.975) # pre-filter target
Y_CLIP_Q   = (0.025, 0.975) # clip on y_train
REQUIRE_COORDS = True       # delete rows without lat,lon

K_DEPOSIT_TO_MONTHLY = 30   # convert credit to rent

BASE_FEATURES = ["building_size", "rooms_count", "age", "utm_x", "utm_y"]
ENGINEERED_FEATURES = ["rooms_per_100sqm"]
ALL_FEATURES = BASE_FEATURES + ENGINEERED_FEATURES

# -------------------- Load & Targets --------------------
clean_data = pd.read_csv('/content/drive/My Drive/Colab Notebooks/Project/df.csv').drop(columns=["Unnamed: 0"], errors="ignore")
clean_data["final_sale_price"] = clean_data["price_value"]
clean_data["final_rent_price"] = clean_data["rent_value"].fillna(0) + clean_data["credit_value"].fillna(0) / K_DEPOSIT_TO_MONTHLY

sale_data = clean_data[clean_data["final_sale_price"].notna()].copy()
rent_data = clean_data[clean_data["final_rent_price"] > 0].copy()

# -------------------- Pre-clean Helpers --------------------
def clip_target_quantile(df, target_col, qlo=0.025, qhi=0.975):
    lo, hi = df[target_col].quantile([qlo, qhi])
    return df[(df[target_col] >= lo) & (df[target_col] <= hi)].copy()

def fill_core_numeric_with_median(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(df[c].median())
    return df

# Remove empty coordinates (if any)
if REQUIRE_COORDS:
    sale_data = sale_data.dropna(subset=["location_longitude_x", "location_latitude_x"]).copy()
    rent_data = rent_data.dropna(subset=["location_longitude_x", "location_latitude_x"]).copy()

# Fill missing values
core_cols = ["building_size", "rooms_count", "construction_year"]
sale_data = fill_core_numeric_with_median(sale_data, core_cols)
rent_data = fill_core_numeric_with_median(rent_data, core_cols)

# sanity checks
sale_data = sale_data[(sale_data["building_size"] > 0) & (sale_data["rooms_count"] > 0)].copy()
rent_data = rent_data[(rent_data["building_size"] > 0) & (rent_data["rooms_count"] > 0)].copy()

# Sales target = price per square meter
sale_data = sale_data[sale_data["building_size"] > 0].copy()
sale_data["price_per_sqm"] = sale_data["final_sale_price"] / sale_data["building_size"]

# pre-clip on target
sale_data = clip_target_quantile(sale_data, "price_per_sqm", PRE_CLIP_Q[0], PRE_CLIP_Q[1])
rent_data = clip_target_quantile(rent_data, "final_rent_price", PRE_CLIP_Q[0], PRE_CLIP_Q[1])

# -------------------- Feature Builders --------------------
def add_age_feature(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["age"] = CURRENT_YEAR_SHAMSI - out["construction_year"]
    return out

def latlon_to_utm(row: pd.Series):
    try:
        lon = row["location_longitude_x"]
        lat = row["location_latitude_x"]
        if pd.isna(lon) or pd.isna(lat):
            return pd.Series([None, None])
        zone = int((lon + 180) / 6) + 1
        epsg = f"326{zone:02d}" if lat >= 0 else f"327{zone:02d}"
        transformer = Transformer.from_crs("epsg:4326", f"epsg:{epsg}", always_xy=True)
        utm_x, utm_y = transformer.transform(lon, lat)
        return pd.Series([utm_x, utm_y])
    except Exception:
        return pd.Series([None, None])

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_age_feature(df)
    df[["utm_x", "utm_y"]] = df.apply(latlon_to_utm, axis=1)
    for c in ["building_size", "rooms_count", "age", "utm_x", "utm_y"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    eps = 1e-6
    df["rooms_per_100sqm"] = (df["rooms_count"] / (df["building_size"].abs() + eps)) * 100
    df["rooms_per_100sqm"] = pd.to_numeric(df["rooms_per_100sqm"], errors="coerce")
    return df

def prune_bad_features(df: pd.DataFrame, candidate_cols, max_null_ratio=0.95, min_unique=2):
    kept, dropped = [], []
    for c in candidate_cols:
        if c not in df.columns:
            dropped.append((c, "missing_column"))
            continue
        null_ratio = df[c].isna().mean()
        nunq = df[c].nunique(dropna=True)
        if null_ratio > max_null_ratio:
            dropped.append((c, f"high_null_ratio={null_ratio:.2f}"))
        elif nunq < min_unique:
            dropped.append((c, f"low_variance nunique={nunq}"))
        else:
            kept.append(c)
    return kept, dropped

def make_preprocessor(feature_cols):
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), feature_cols)], remainder="drop")

def xgb_quantile_objective(tau):
    def obj(preds, dtrain):
        y = dtrain.get_label()
        diff = y - preds
        sign = (diff > 0).astype(np.float32)
        grad = -tau * sign + (1 - tau) * (1 - sign)
        hess = np.full_like(grad, 1e-6)
        return grad, hess
    return obj

# -------------------- Pipelines --------------------
def build_rf_pipeline(feature_cols):
    pre = make_preprocessor(feature_cols)
    rf = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
    pipe = Pipeline([("prep", pre),
                     ("reg", TransformedTargetRegressor(regressor=rf, func=np.log1p, inverse_func=np.expm1))])
    param_space = {
        "reg__regressor__n_estimators": [200, 400, 600],
        "reg__regressor__max_depth": [None, 10, 14],
        "reg__regressor__min_samples_leaf": [1, 2, 3],
        "reg__regressor__min_samples_split": [2, 5],
        "reg__regressor__max_features": ["sqrt", "log2"],
        "reg__regressor__bootstrap": [True],
    }
    return pipe, param_space

def build_xgb_pipeline(feature_cols):
    pre = make_preprocessor(feature_cols)
    if USE_QUANTILE and abs(QUANTILE_TAU - 0.5) < 1e-9:
        objective, custom_obj = "reg:absoluteerror", None
    elif USE_QUANTILE:
        objective, custom_obj = None, xgb_quantile_objective(QUANTILE_TAU)
    else:
        objective, custom_obj = "reg:squarederror", None

    xgbr = xgb.XGBRegressor(objective=objective, tree_method="hist", random_state=RANDOM_SEED, n_jobs=-1)
    pipe = Pipeline([("prep", pre),
                     ("reg", TransformedTargetRegressor(regressor=xgbr, func=np.log1p, inverse_func=np.expm1))])
    param_space = {
        "reg__regressor__n_estimators": [300, 500, 800, 1000],
        "reg__regressor__learning_rate": np.linspace(0.03, 0.2, 6),
        "reg__regressor__max_depth": [3, 4, 5, 6, 8],
        "reg__regressor__min_child_weight": [1, 3, 5, 7],
        "reg__regressor__subsample": [0.7, 0.85, 1.0],
        "reg__regressor__colsample_bytree": [0.7, 0.85, 1.0],
        "reg__regressor__reg_lambda": np.logspace(-3, 1, 5),
        "reg__regressor__reg_alpha": [0.0, 0.001, 0.01, 0.1],
    }
    return pipe, param_space, custom_obj

# -------------------- Train/Eval --------------------
def train_model_with_validation(df: pd.DataFrame, target_col: str):
    df = df.copy()
    df = df[df[target_col].notna() & (df[target_col] > 0)]
    df = build_features(df)
    cols = [c for c in ALL_FEATURES if c in df.columns]

    if len(df) > SAMPLE_LIMIT:
        df = df.sample(n=SAMPLE_LIMIT, random_state=RANDOM_SEED)

    X, y = df[cols].copy(), df[target_col].copy()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED)
    kept_cols, dropped_cols = prune_bad_features(X_train, cols)

    if MODEL_TYPE.lower() == "rf":
        base_pipe, param_space = build_rf_pipeline(kept_cols)
        n_iter = 10
        fit_kwargs = {}
    elif MODEL_TYPE.lower() == "xgb":
        base_pipe, param_space, custom_obj = build_xgb_pipeline(kept_cols)
        n_iter = 12
        fit_kwargs = {}
        if USE_QUANTILE and abs(QUANTILE_TAU - 0.5) >= 1e-9:
            fit_kwargs["reg__regressor__obj"] = custom_obj
    else:
        raise ValueError("MODEL_TYPE must be 'rf' or 'xgb'")

    kfold = KFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    search = RandomizedSearchCV(estimator=base_pipe,
                                param_distributions=param_space,
                                n_iter=n_iter, cv=kfold,
                                scoring="neg_mean_absolute_error",
                                n_jobs=-1, random_state=RANDOM_SEED,
                                verbose=0, pre_dispatch="2*n_jobs",
                                error_score="raise")

    qlo, qhi = np.quantile(y_train.dropna(), Y_CLIP_Q)
    y_train_clip = y_train.clip(qlo, qhi)

    search.fit(X_train[kept_cols], y_train_clip, **fit_kwargs)
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test[kept_cols])

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)
    medae = median_absolute_error(y_test, y_pred)

    report = {
        "model_type": MODEL_TYPE.lower(),
        "best_params_from_cv": search.best_params_,
        "cv_mean_negMAE": float(search.best_score_),
        "test_MAE": float(mae),
        "test_MedAE": float(medae),
        "test_RMSE": float(rmse),
        "test_R2": float(r2),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "used_features": kept_cols,
        "dropped_features": dropped_cols
    }
    return best_model, report

# -------------------- Manager class --------------------
class SimplePriceModels:
    def __init__(self):
        self.sale_model = None
        self.sale_report = None
        self.rent_model = None
        self.rent_report = None

    def fit_sale(self, df_sale: pd.DataFrame):
        model, report = train_model_with_validation(df_sale, target_col="price_per_sqm")
        self.sale_model, self.sale_report = model, report
        return report

    def fit_rent(self, df_rent: pd.DataFrame):
        model, report = train_model_with_validation(df_rent, target_col="final_rent_price")
        self.rent_model, self.rent_report = model, report
        return report

    def _prepare_features_for_predict(self, df_new: pd.DataFrame, used_cols):
        df_new = build_features(df_new)
        missing = [c for c in used_cols if c not in df_new.columns]
        if missing:
            raise ValueError(f"Missing required features for prediction: {missing}")
        return df_new[used_cols]

    def predict_sale(self, df_new: pd.DataFrame, return_per_sqm: bool = False):
        if self.sale_model is None:
            raise ValueError("Sale model is not trained yet. Call fit_sale first.")
        used_cols = self.sale_report["used_features"]
        X_new = self._prepare_features_for_predict(df_new, used_cols)
        pred_per_sqm = self.sale_model.predict(X_new)
        if return_per_sqm:
            return pred_per_sqm
        sizes = pd.to_numeric(df_new["building_size"], errors="coerce").fillna(0).to_numpy()
        return pred_per_sqm * sizes

    def predict_rent(self, df_new: pd.DataFrame):
        if self.rent_model is None:
            raise ValueError("Rent model is not trained yet. Call fit_rent first.")
        used_cols = self.rent_report["used_features"]
        X_new = self._prepare_features_for_predict(df_new, used_cols)
        return self.rent_model.predict(X_new)



In [ ]:
# ------------------- RF -------------------
MODEL_TYPE = "rf"   # RandomForest

print("========== RandomForest ==========\n")
models_rf = SimplePriceModels()

sale_report_rf = models_rf.fit_sale(sale_data)
rent_report_rf = models_rf.fit_rent(rent_data)

print("sale model report (RF):\n")
print(sale_report_rf)

print("\n rent model report (RF):\n")
print(rent_report_rf)


# ------------------- XGB -------------------
MODEL_TYPE = "xgb"  # XGBoost

print("\n========== XGBoost ==========\n")
models_xgb = SimplePriceModels()

sale_report_xgb = models_xgb.fit_sale(sale_data)
rent_report_xgb = models_xgb.fit_rent(rent_data)

print("sale model report (XGB):\n")
print(sale_report_xgb)

print("\nrent model report (XGB):\n")
print(rent_report_xgb)


print(f"\n[Model: XGB | USE_QUANTILE={USE_QUANTILE} | TAU={QUANTILE_TAU}]")


========== RandomForest ==========

sale model report (RF):

{'model_type': 'rf', 'best_params_from_cv': {'reg__regressor__n_estimators': 600, 'reg__regressor__min_samples_split': 5, 'reg__regressor__min_samples_leaf': 2, 'reg__regressor__max_features': 'sqrt', 'reg__regressor__max_depth': None, 'reg__regressor__bootstrap': True}, 'cv_mean_negMAE': -12127251.50624164, 'test_MAE': 12079021.576395405, 'test_MedAE': 6189245.8982883375, 'test_RMSE': 21477478.584413715, 'test_R2': 0.6034962133139035, 'n_train': 16000, 'n_test': 4000, 'used_features': ['building_size', 'rooms_count', 'age', 'utm_x', 'utm_y', 'rooms_per_100sqm'], 'dropped_features': []}

 rent model report (RF):

{'model_type': 'rf', 'best_params_from_cv': {'reg__regressor__n_estimators': 400, 'reg__regressor__min_samples_split': 5, 'reg__regressor__min_samples_leaf': 1, 'reg__regressor__max_features': 'sqrt', 'reg__regressor__max_depth': None, 'reg__regressor__bootstrap': True}, 'cv_mean_negMAE': -6492987.227202049, 'test_MA